# `cs336_basics/model.py` 代码式复习笔记

这版 notebook 不再只是“解释概念”，而是改成 **markdown 讲思路，code cell 放完整实现** 的形式。你可以把它当成一份从上到下逐步把 `model.py` 写出来的草稿。

代码 cell 里的实现与 `model.py` 保持同一条计算逻辑；markdown 负责解释两件事：

- 下一步为什么自然会写出这段代码。
- 这一段代码在整个 Transformer 前向传播里处在什么位置。

## 总路线

整个语言模型的主链是：

$$
\text{token ids}
\xrightarrow{\text{embedding}}
X_0
\xrightarrow{N\times\text{Transformer block}}
X_L
\xrightarrow{\text{RMSNorm}}
\hat X_L
\xrightarrow{\text{lm head}}
\text{logits}
$$

每个 block 内部又是：

$$
Y = X + \mathrm{MHA\_RoPE}(\mathrm{RMSNorm}(X))
$$

$$
Z = Y + \mathrm{SwiGLU}(\mathrm{RMSNorm}(Y))
$$

所以下面会按“先写最基础的积木，再一路拼到整个模型”的顺序来整理。


In [ ]:
from __future__ import annotations

import torch
from torch import Tensor


## 1. 先定两个最基础的接口：`embedding()` 和 `linear()`

真正开始写模型时，最先应该固定的是 **输入如何进来**，以及 **后续所有投影统一怎么写**。

`embedding()` 负责把离散 token id 变成连续向量；它的本质不是矩阵乘法，而是查表：

$$
E \in \mathbb{R}^{V \times d_{model}}, \quad \mathrm{embedding}(t) = E[t]
$$

`linear()` 则是后面几乎所有子模块都要复用的基础操作。这个项目里权重统一存成 `(d_out, d_in)`，所以真正计算时要写成：

$$
Y = X W^T
$$

写这两个函数时，脑子里只要抓住两件事：

- `embedding` 是“索引”。
- `linear` 是“对最后一维做线性变换”。


In [ ]:
def linear(weights: Tensor, in_features: Tensor) -> Tensor:
    return in_features @ weights.T


def embedding(weights: Tensor, token_ids: Tensor) -> Tensor:
    return weights[token_ids]


## 2. 再把局部积木写出来：`silu()`、`rmsnorm()`、`swiglu()`

写完输入层和统一投影后，下一个自然步骤是把 block 里会反复调用的小模块补齐。

### `silu()`

这是最简单的逐元素激活：

$$
\mathrm{SiLU}(x) = x \cdot \sigma(x)
$$

所以代码不会复杂，直接按公式翻译即可。

### `rmsnorm()`

这里的思考顺序通常是：

1. 只沿最后一维归一化，因为最后一维才是特征维。
2. 为了广播方便，`mean(..., keepdim=True)` 不能丢维度。
3. 半精度下平方、均值、开根号容易不稳，所以先升到 `float32`。
4. 最后再转回输入 dtype。

它对应的公式是：

$$
\mathrm{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2 + \varepsilon}
$$

$$
\mathrm{RMSNorm}(x)=\frac{x}{\mathrm{RMS}(x)} \odot w
$$

### `swiglu()`

这一步其实就是 FFN 的完整写法：

$$
\mathrm{SwiGLU}(x)=W_2\big(\mathrm{SiLU}(W_1x) \odot W_3x\big)
$$

也就是说，先写两路投影，再门控相乘，最后投回 `d_model`。


In [ ]:
def silu(in_features: Tensor) -> Tensor:
    return in_features * torch.sigmoid(in_features)


def rmsnorm(in_features: Tensor, weights: Tensor, eps: float = 1e-5) -> Tensor:
    in_temp = in_features.to(torch.float32)
    rms = torch.sqrt(in_temp.pow(2).mean(dim=-1, keepdim=True) + eps)
    y = in_temp / rms * weights.to(torch.float32)
    return y.to(in_features.dtype)


def swiglu(
    in_features: Tensor,
    w1_weight: Tensor,
    w2_weight: Tensor,
    w3_weight: Tensor,
) -> Tensor:
    gate = silu(linear(w1_weight, in_features))
    value = linear(w3_weight, in_features)
    return linear(w2_weight, gate * value)


## 3. 写 attention 之前，先把形状问题解决：`split_heads()`、`merge_heads()`、mask、scaled attention

到这一步，最容易卡住的不是公式，而是 shape。

输入 token 表示通常是：

$$
(..., seq\_len, d_{model})
$$

但多头注意力真正想要的内部形状是：

$$
(..., num\_heads, seq\_len, d_{head})
$$

所以 `split_heads()` 的心智模型很简单：

1. 先把最后一维拆成 `(num_heads, head_dim)`。
2. 再把 head 维交换到 sequence 前面。

`merge_heads()` 则完全反过来。

另外，真正写 attention 核心前，还要先补两件事：

- `build_causal_mask()`：构造下三角 mask，让当前位置只能看自己和过去。
- `scaled_dot_product_attention()`：实现 `softmax(QK^T / sqrt(d_k))V`。

这里有一个容易写错的点：`K` 不能用 `K.T`，因为张量经常不止二维；这里只想交换最后两维，所以必须写 `K.transpose(-2, -1)`。


In [ ]:
def build_causal_mask(sequence_length: int, device: torch.device | str) -> Tensor:
    return torch.tril(
        torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=device)
    )


def scaled_dot_product_attention(
    Q: Tensor,
    K: Tensor,
    V: Tensor,
    mask: Tensor | None = None,
) -> Tensor:
    d_k = Q.shape[-1]
    att_logits = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)

    if mask is not None:
        att_logits = att_logits.masked_fill(~mask, float("-inf"))

    att_weights = torch.softmax(att_logits, dim=-1)

    if mask is not None:
        att_weights = att_weights.masked_fill(~mask, 0.0)

    return att_weights @ V


def split_heads(x: Tensor, num_heads: int) -> Tensor:
    d_model = x.shape[-1]
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")

    head_dim = d_model // num_heads
    x = x.reshape(*x.shape[:-1], num_heads, head_dim)
    return x.transpose(-3, -2)


def merge_heads(x: Tensor) -> Tensor:
    num_heads = x.shape[-3]
    head_dim = x.shape[-1]
    x = x.transpose(-3, -2).contiguous()
    return x.reshape(*x.shape[:-2], num_heads * head_dim)


## 4. 然后写 RoPE：先校验，再广播，再旋转

RoPE 是这份作业里最值得慢一点写的函数，因为它既有数学结构，也有广播细节。

它不是把位置向量加到表示上，而是把最后一维两两配对后做二维旋转：

$$
(x_0, x_1), (x_2, x_3), \dots
$$

对第 `i` 对维度，频率是：

$$
\omega_i = \theta^{-2i/d_k}
$$

对位置 `p`，角度是：

$$
\varphi_{p, i} = p \cdot \omega_i
$$

然后旋转：

$$
\begin{bmatrix}
x'_{2i} \\
x'_{2i+1}
\end{bmatrix}
=
\begin{bmatrix}
\cos\varphi_{p,i} & -\sin\varphi_{p,i} \\
\sin\varphi_{p,i} & \cos\varphi_{p,i}
\end{bmatrix}
\begin{bmatrix}
x_{2i} \\
x_{2i+1}
\end{bmatrix}
$$

真正开始写代码时，最稳的顺序是：

1. 先检查最后一维能不能两两配对，所以 `d_k` 必须是偶数。
2. 再检查 `token_positions` 的长度是不是和 `seq_len` 对齐。
3. 把位置张量一路 `unsqueeze` 到能和 `(..., num_heads, seq_len, head_dim)` 广播兼容。
4. 最后才做奇偶拆分、旋转、再拼回去。

这里 `while positions.ndim < x.ndim - 1` 是关键，因为 notebook 想保留和 `model.py` 一样的通用性，而不把实现写死成只支持某一种 batch 维度。


In [ ]:
def apply_rope(
    in_query_or_key: Tensor,
    token_positions: Tensor,
    theta: float,
    max_seq_len: int,
) -> Tensor:
    x = in_query_or_key
    d_k = x.shape[-1]

    if d_k % 2 != 0:
        raise ValueError(f"d_k must be even, got {d_k}")

    if token_positions.shape[-1] != x.shape[-2]:
        raise ValueError(
            f"token_positions last dimension must match seq_len: "
            f"{token_positions.shape[-1]} vs {x.shape[-2]}"
        )

    if token_positions.numel() > 0:
        min_pos = int(token_positions.min().item())
        max_pos = int(token_positions.max().item())
        if min_pos < 0 or max_pos >= max_seq_len:
            raise ValueError(
                f"token positions must be in [0, {max_seq_len - 1}], "
                f"got min={min_pos}, max={max_pos}"
            )

    compute_dtype = (
        torch.float32
        if x.dtype in (torch.float16, torch.bfloat16)
        else x.dtype
    )

    half_dim = d_k // 2

    positions = token_positions.to(device=x.device, dtype=compute_dtype)
    while positions.ndim < x.ndim - 1:
        positions = positions.unsqueeze(-2)
    positions = positions.unsqueeze(-1)

    i = torch.arange(half_dim, device=x.device, dtype=compute_dtype)
    inv_freq = theta ** (-2.0 * i / d_k)
    angles = positions * inv_freq

    cos_angles = torch.cos(angles)
    sin_angles = torch.sin(angles)

    x_even = x[..., 0::2].to(dtype=compute_dtype)
    x_odd = x[..., 1::2].to(dtype=compute_dtype)

    out_even = x_even * cos_angles - x_odd * sin_angles
    out_odd = x_even * sin_angles + x_odd * cos_angles

    out = torch.stack((out_even, out_odd), dim=-1).flatten(-2)
    return out.to(dtype=x.dtype)


## 5. 把注意力整条链串起来：`multihead_self_attention()` 和 `multihead_self_attention_with_rope()`

到这里其实就只剩下“按顺序把前面的 helper 接起来”。

普通版多头自注意力的顺序是：

$$
X \xrightarrow{W_Q, W_K, W_V} Q, K, V
\xrightarrow{split\ heads}
\xrightarrow{causal\ attention}
\xrightarrow{merge\ heads}
\xrightarrow{W_O}
Y
$$

带 RoPE 的版本只多了一步：

$$
Q, K \xrightarrow{\mathrm{RoPE}} \tilde Q, \tilde K
$$

然后再去做 attention。

最值得记住的实现细节只有两个：

- RoPE 应该发生在 `split_heads()` 之后，因为旋转维度是 `d_head`，不是整个 `d_model`。
- 如果外面没传 `token_positions`，最自然的默认值就是 `0, 1, ..., seq_len - 1`。


In [ ]:
def multihead_self_attention(
    d_model: int,
    num_heads: int,
    q_proj_weight: Tensor,
    k_proj_weight: Tensor,
    v_proj_weight: Tensor,
    o_proj_weight: Tensor,
    in_features: Tensor,
) -> Tensor:
    seq_len = in_features.shape[-2]
    d_model = in_features.shape[-1]
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")

    Q = split_heads(linear(q_proj_weight, in_features), num_heads)
    K = split_heads(linear(k_proj_weight, in_features), num_heads)
    V = split_heads(linear(v_proj_weight, in_features), num_heads)

    mask = build_causal_mask(seq_len, in_features.device)
    attn_out = scaled_dot_product_attention(Q, K, V, mask=mask)
    attn_out = merge_heads(attn_out)
    return linear(o_proj_weight, attn_out)


def multihead_self_attention_with_rope(
    in_features: Tensor,
    q_proj_weight: Tensor,
    k_proj_weight: Tensor,
    v_proj_weight: Tensor,
    o_proj_weight: Tensor,
    num_heads: int,
    theta: float,
    max_seq_len: int,
    token_positions: Tensor | None = None,
) -> Tensor:
    seq_len = in_features.shape[-2]
    d_model = in_features.shape[-1]
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")

    if token_positions is None:
        token_positions = torch.arange(seq_len, device=in_features.device)

    Q = split_heads(linear(q_proj_weight, in_features), num_heads)
    K = split_heads(linear(k_proj_weight, in_features), num_heads)
    V = split_heads(linear(v_proj_weight, in_features), num_heads)

    Q = apply_rope(Q, token_positions, theta, max_seq_len)
    K = apply_rope(K, token_positions, theta, max_seq_len)

    mask = build_causal_mask(seq_len, in_features.device)
    attn_out = scaled_dot_product_attention(Q, K, V, mask=mask)
    attn_out = merge_heads(attn_out)
    return linear(o_proj_weight, attn_out)


## 6. 最后把 block 和整个语言模型拼起来

这一步其实是在把前面所有积木按架构图落地。

单层 block 是标准的 pre-norm 结构：

$$
Y = X + \mathrm{MHA\_RoPE}(\mathrm{RMSNorm}(X))
$$

$$
Z = Y + \mathrm{SwiGLU}(\mathrm{RMSNorm}(Y))
$$

所以写 `transformer_block()` 时，基本就是照着上面两行翻译成代码。

最顶层的 `transformer_lm()` 也同理：

1. 先查 token embedding。
2. 用 `for block in block_weights` 反复过每一层。
3. 最后补一次 final RMSNorm。
4. 再用 LM head 投到词表维度，得到 logits。

这里返回的是 **logits**，不是概率。原因很简单：训练时交叉熵损失内部会自己做 softmax，所以前向函数不应该提前把输出压成概率。


In [ ]:
def transformer_block(
    in_features: Tensor,
    *,
    ln1_weight: Tensor,
    q_proj_weight: Tensor,
    k_proj_weight: Tensor,
    v_proj_weight: Tensor,
    o_proj_weight: Tensor,
    ln2_weight: Tensor,
    w1_weight: Tensor,
    w2_weight: Tensor,
    w3_weight: Tensor,
    num_heads: int,
    theta: float,
    max_seq_len: int,
) -> Tensor:
    x = in_features

    attn_input = rmsnorm(x, ln1_weight)
    attn_output = multihead_self_attention_with_rope(
        attn_input,
        q_proj_weight,
        k_proj_weight,
        v_proj_weight,
        o_proj_weight,
        num_heads,
        theta,
        max_seq_len=max_seq_len,
    )
    x = x + attn_output

    ffn_input = rmsnorm(x, ln2_weight)
    ffn_output = swiglu(ffn_input, w1_weight, w2_weight, w3_weight)
    x = x + ffn_output
    return x


def transformer_lm(
    in_indices: Tensor,
    *,
    token_embedding_weight: Tensor,
    block_weights: list[dict[str, Tensor]],
    ln_final_weight: Tensor,
    lm_head_weight: Tensor,
    num_heads: int,
    theta: float,
    max_seq_len: int,
) -> Tensor:
    x = embedding(token_embedding_weight, in_indices)

    for block in block_weights:
        x = transformer_block(
            x,
            **block,
            num_heads=num_heads,
            theta=theta,
            max_seq_len=max_seq_len,
        )

    x = rmsnorm(x, ln_final_weight)
    return linear(lm_head_weight, x)


__all__ = [
    "linear",
    "embedding",
    "silu",
    "rmsnorm",
    "swiglu",
    "build_causal_mask",
    "scaled_dot_product_attention",
    "split_heads",
    "merge_heads",
    "apply_rope",
    "multihead_self_attention",
    "multihead_self_attention_with_rope",
    "transformer_block",
    "transformer_lm",
]


## 7. 复习时，建议按这条顺序回放代码

如果你以后要自己重新把这份 `model.py` 写一遍，最稳的回放顺序就是：

1. 先写统一接口：`embedding`、`linear`。
2. 再写局部积木：`silu`、`rmsnorm`、`swiglu`。
3. 再处理注意力的 shape：`split_heads`、`merge_heads`、`build_causal_mask`、`scaled_dot_product_attention`。
4. 然后单独攻克 `apply_rope`。
5. 再把整条注意力链拼成 `multihead_self_attention_with_rope`。
6. 最后写 `transformer_block` 和 `transformer_lm`。

这样复习时你看到的就不再是一堆分散的小函数，而是一条很清楚的构造链：

`token ids -> embedding -> qkv projections -> split heads -> rope -> causal attention -> merge heads -> output projection -> residual -> ffn -> logits`

notebook 现在的结构，就是为了让这条链条尽量显出来。
